# Notebook 1 — Chargement des données
## Assistant Juridique RAG — Code du Travail

### Objectif
Ce notebook couvre la première étape du pipeline RAG :
le chargement et l'exploration des PDFs du Code du Travail.

### Contenu
1. Installation des dépendances
2. Configuration de la clé API
3. Montage Google Drive
4. Chargement du PDF avec PyPDFLoader
5. Exploration et analyse du contenu chargé

In [2]:
!pip install langchain langchain-openai langchain-community pypdf chromadb python-dotenv -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

## Étape 1 — Configuration de la clé API OpenAI
La clé est stockée dans les Secrets Colab

In [3]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Clé API :", "OK" if os.environ.get("OPENAI_API_KEY") else "MANQUANTE")

Clé API : OK


## Étape 2 — Google Drive
Les PDFs sont stockés sur Google Drive partagé.
Chemin : Mon Drive/GEN_AI_Assistant_Juridique/data/

In [5]:
from pathlib import Path

DOCS_DIR = "/content/drive/MyDrive/GEN_AI_Assistant_Juridique/data"

# Liste tous les PDFs disponibles
pdf_files = list(Path(DOCS_DIR).glob("*.pdf"))

print(f"✅ {len(pdf_files)} fichiers PDF trouvés :\n")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

✅ 7 fichiers PDF trouvés :

  - Salaires_avantages.pdf
  - Syndicats.pdf
  - Fin_de_contrat.pdf
  - Harcelement.pdf
  - Contrat_de_travail.pdf
  - Egalites_professionnelles.pdf
  - sécurité_au_travail.pdf


## Étape 3 — Chargement du PDF avec PyPDFLoader



In [6]:
from langchain_community.document_loaders import PyPDFLoader

docs = []
for pdf_path in pdf_files:
    print(f"Chargement : {pdf_path.name}")
    loader = PyPDFLoader(str(pdf_path))
    docs.extend(loader.load())

print(f"\n✅ Total : {len(docs)} pages chargées depuis {len(pdf_files)} fichiers")

Chargement : Salaires_avantages.pdf
Chargement : Syndicats.pdf
Chargement : Fin_de_contrat.pdf
Chargement : Harcelement.pdf
Chargement : Contrat_de_travail.pdf
Chargement : Egalites_professionnelles.pdf
Chargement : sécurité_au_travail.pdf

✅ Total : 479 pages chargées depuis 7 fichiers


## Étape 4 — Analyse du contenu chargé

On analyse la structure des documents chargés :
longueur des pages, répartition par fichier,
qualité du texte extrait.

In [7]:
import numpy as np
from collections import Counter

# Analyse globale
longueurs = [len(doc.page_content) for doc in docs]

print("=== Analyse globale ===")
print(f"Total pages      : {len(docs)}")
print(f"Longueur moyenne : {int(np.mean(longueurs))} caractères")
print(f"Longueur min     : {min(longueurs)} caractères")
print(f"Longueur max     : {max(longueurs)} caractères")

# Répartition par fichier
print("\n=== Pages par fichier ===")
sources = [doc.metadata['source'].split('/')[-1] for doc in docs]
for fichier, count in Counter(sources).items():
    print(f"  {fichier} : {count} pages")

=== Analyse globale ===
Total pages      : 479
Longueur moyenne : 3582 caractères
Longueur min     : 395 caractères
Longueur max     : 5793 caractères

=== Pages par fichier ===
  Salaires_avantages.pdf : 169 pages
  Syndicats.pdf : 34 pages
  Fin_de_contrat.pdf : 25 pages
  Harcelement.pdf : 6 pages
  Contrat_de_travail.pdf : 164 pages
  Egalites_professionnelles.pdf : 6 pages
  sécurité_au_travail.pdf : 75 pages


## Étape 5 — Vérification de la qualité du texte

On vérifie que le texte extrait est lisible
et contient bien des articles du Code du Travail.

In [8]:
import random

print("=== Vérification sur 5 pages aléatoires ===\n")
pages_test = random.sample(range(len(docs)), 5)

for i in pages_test:
    source = docs[i].metadata['source'].split('/')[-1]
    page = docs[i].metadata.get('page', '?')
    print(f"--- Fichier : {source} | Page {page} ---")
    print(docs[i].page_content[:300])
    print()

=== Vérification sur 5 pages aléatoires ===

--- Fichier : Contrat_de_travail.pdf | Page 141 ---
Partie législative - Première partie : Les relations individuelles de travail - Livre II : Le contrat de travail
L. 1254-7  ORDONNANCE n°2015-380 du 2 avril 2015 - art. 2      
  Legif.   
  Plan   
  Jp.Judi.   
  Jp.Admin.   
  Juricaf
Le contrat de travail est conclu entre l'entreprise de portage

--- Fichier : Syndicats.pdf | Page 14 ---
Partie législative - Deuxième partie : Les relations collectives de travail - Livre Ier : Les syndicats professionnels
réglementaire, de façon uniforme pour les organisations syndicales de salariés et en fonction de l'audience pour
les organisations professionnelles d'employeurs. Pour l'appréciation

--- Fichier : sécurité_au_travail.pdf | Page 46 ---
Partie législative - Quatrième partie : Santé et sécurité au travail - Livre VI : Institutions et organismes de prévention
Livre VI : Institutions et organismes de prévention
Titre II : Services de préve